In [ ]:
import copy
import pprint
import time
from collections import Counter
import torch
import torch.nn as nn
import wandb
import importlib
import src.cnn.cnn_utils as cnn_utils
import src.cnn.cnn_registry as cnn_registry
import src.cnn.cnn_models as cnn_models
import src.cnn.configs as configs
import src.cnn.cnn_paths as cnn_paths
import src.cnn.cnn_gradcam as cnn_gradcam


importlib.reload(cnn_utils)
#importlib.reload(cnn_gradcam)

# Shallow CNN
## Config

In [ ]:
# see configs.py
cfg = configs.ExperimentConfig()

# =========================================================
# DATASET CONFIG
# =========================================================
cfg.dataset.dataset_dir = cnn_paths.DATASET_DIR  # see cnn_paths.py
cfg.dataset.train_subdir = "train"
cfg.dataset.val_subdir = "validate"
cfg.dataset.test_subdir = None  # or "test" if you have it
cfg.dataset.image_size = 224

# leave transforms as None -> default Resize + ToTensor pipeline
cfg.dataset.train_transform = None
cfg.dataset.eval_transform = None

# optional normalization
cfg.dataset.normalize_mean = None
cfg.dataset.normalize_std = None


# =========================================================
# MODEL CONFIG
# IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
# use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
# =========================================================
cfg.model.name = "shallow_model"
cfg.model.kwargs = {
    "in_channels": 3,
    "num_classes": 10,
    "units": 128,
    "drop": 0.5,
}

# =========================================================
# TRAIN CONFIG
# =========================================================
cfg.train.epochs = 30
cfg.train.device = str(cnn_utils.get_device("auto"))
cfg = cnn_utils.configure_runtime_defaults(cfg)
cfg.train.non_blocking = True
cfg.train.use_amp = False

# RECOMMENDED FOR CUDA
# cfg.train.use_amp = True

cfg.train.grad_clip_norm = None
cfg.train.best_metric = "val/accuracy"
cfg.train.best_mode = "max"
cfg.train.seed = 13

# =========================================================
# DATALOADER CONFIG
# =========================================================
cfg.loader.batch_size = 16
cfg.loader.num_workers = 0
cfg.loader.pin_memory = (torch.device(cfg.train.device).type == "cuda")

# RECOMMENDED FOR CUDA
# cfg.loader.pin_memory = True

cfg.loader.train_shuffle = True
cfg.loader.eval_shuffle = False
cfg.loader.drop_last_train = False
cfg.loader.drop_last_eval = False


# =========================================================
# LOSS CONFIG
# =========================================================
cfg.loss.cls = nn.CrossEntropyLoss
cfg.loss.kwargs = {}


# =========================================================
# OPTIMIZER CONFIG
# =========================================================
cfg.optimizer.cls = torch.optim.Adam
cfg.optimizer.kwargs = {
    "lr": 1e-3,
    "weight_decay": 1e-4,
}


# =========================================================
# SCHEDULER CONFIG
# Example: Reduce LR when validation loss plateaus
# =========================================================
cfg.scheduler.cls = torch.optim.lr_scheduler.ReduceLROnPlateau
cfg.scheduler.kwargs = {
    "mode": "min",
    "factor": 0.5,
    "patience": 2,
}
cfg.scheduler.step_metric = "val/loss"

# =========================================================
# W&B CONFIG
# =========================================================
cfg.wandb.enabled = True
cfg.wandb.project = "MPW-CNN"
cfg.wandb.entity = "MSE_DeLearn_SPR26"
cfg.wandb.mode = "online"  # "online", "offline", or "disabled" for no logging
cfg.wandb.log_confusion_matrix = True

# NOTE: use meaningful names for runs, so the difference is clear
cfg.wandb.run_name = "gradcam_" + time.strftime("%Y%m%d-%H%M%S")

# NOTE: use meaningful grouping, for example, by model or by task.
# E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
cfg.wandb.group = "gradcam"
cfg.wandb.job_type = "train"

# NOTE: use meaningful tags to filter runs in UI
cfg.wandb.tags = ["cnn", "gradcam"]
cfg.wandb.notes = ""

cfg.wandb.log_epoch_metrics = True
cfg.wandb.log_every_n_epochs = 1

cfg.wandb.metric_allowlist = {
    "train/loss",
    "train/accuracy",
    "val/loss",
    "val/accuracy",
    "gap/accuracy",
    "gap/loss",
    "lr",
}

cfg.wandb.summary_allowlist = {
    "best_epoch",
    "best_metric_name",
    "best_metric_value",
    "train_final/loss",
    "train_final/accuracy",
    "val_final/loss",
    "val_final/accuracy",
}

cfg.wandb.watch_model = False
cfg.wandb.watch_log = "all"
cfg.wandb.watch_log_freq = 100

# Data and loaders


In [ ]:
datasets_dict = cnn_utils.load_datasets(cfg.dataset)

train_dataset = datasets_dict["train"]
val_dataset = datasets_dict["val"]
test_dataset = datasets_dict.get("test")

In [ ]:
train_loader = cnn_utils.make_train_loader(train_dataset, cfg)
val_loader = cnn_utils.make_eval_loader(val_dataset, cfg)

test_loader = None
if test_dataset is not None:
    test_loader = cnn_utils.make_eval_loader(test_dataset, cfg)

# Model

In [ ]:
# Model builder from config

model = cnn_registry.build_model(cfg.model)

print(model)
print(f"Trainable parameters: {cnn_utils.get_num_parameters(model):,}")

## Train

In [ ]:
model, history, result = cnn_utils.train_and_evaluate_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    cfg=cfg,
    run_name=cfg.wandb.run_name,
)

## Evaluate and print results

In [ ]:
cfg_eval = copy.deepcopy(cfg)
cfg_eval.train.device = "cpu"

model_cpu = model.to("cpu").eval()

if torch.backends.mps.is_available():
    torch.mps.synchronize()
    torch.mps.empty_cache()

loader_for_cm = test_loader if test_loader is not None else val_loader
split_name = "test" if test_loader is not None else "val"
class_names = loader_for_cm.dataset.classes

y_true, y_pred = cnn_utils.predict_loader(model_cpu, loader_for_cm, cfg_eval)

print("structure of predictions/true labels:")
print("true label counts:", Counter(y_true))
print("pred label counts:", Counter(y_pred))
print("class_names:", class_names)

print("Unique y_true:", sorted(set(y_true)))
print("Unique y_pred:", sorted(set(y_pred)))

In [ ]:
pprint.pprint(result)

## Save model

In [ ]:
ckpt_path = cnn_paths.CKPT_DIR  # in data/checkpoints/

# Checkpoint file with timestamp of when the training was completed (in train_eval)
checkpoint_file = str(cfg.wandb.run_name) + '_' + result["timestamp"] + ".pth"

cnn_utils.save_checkpoint(
    path=ckpt_path,
    filename=checkpoint_file,
    model=model,
    cfg=cfg,
    extra={"result": result},
)

## Load model from file (example)

In [ ]:
model, checkpoint = cnn_utils.load_model_for_inference(
    path=ckpt_path,
    filename=checkpoint_file,)

## Log checkpoint

In [ ]:
if cfg.wandb.enabled and cfg.wandb.mode != "disabled":
    with wandb.init(
        project=cfg.wandb.project,
        entity=cfg.wandb.entity,
        job_type="log_checkpoint",
        #name=f"{cfg.wandb.run_name}_artifact",
        mode=cfg.wandb.mode,
        id=result["wandb_run_id"], # add to the same run!
        resume="must",              # for that, run is resumed
    ):
        cnn_utils.log_checkpoint_artifact(
            path=ckpt_path,
            filename=checkpoint_file,
            cfg=cfg,
            result=result,
        )

## Log CF to W&B

In [ ]:
fig_counts = cnn_utils.make_confusion_matrix_fig(
    y_true=y_true,
    y_pred=y_pred,
    class_names=class_names,
    normalize=None,
    title=f"{split_name.capitalize()} Confusion Matrix (counts)",
)

fig_norm = cnn_utils.make_confusion_matrix_fig(
    y_true=y_true,
    y_pred=y_pred,
    class_names=class_names,
    normalize="true",
    title=f"{split_name.capitalize()} Confusion Matrix (normalized)",
)

with wandb.init(
    project=cfg.wandb.project,
    entity=cfg.wandb.entity,
    mode=cfg.wandb.mode,
    id=result["wandb_run_id"], # add to the same run!
    resume="must",              # for that, run is resumed
    job_type="log_confusion_matrix",
):
    try:
        cnn_utils.log_confusion_matrix(
            y_true=y_true,
            y_pred=y_pred,
            class_names=class_names,
            key=f"{split_name}/confusion_matrix_chart",
            title=f"{split_name.capitalize()} Confusion Matrix",
            split_table=False,
        )
    except Exception as e:
        wandb.log({
            f"{split_name}/confusion_matrix_chart_error": str(e)
        })
        print("W&B custom confusion matrix failed:", e)

    wandb.log({
        f"{split_name}/confusion_matrix_counts": wandb.Image(fig_counts),
        f"{split_name}/confusion_matrix_normalized": wandb.Image(fig_norm),
    })

    wandb.finish()

# Grad-CAM

In [ ]:
%load_ext autoreload
%autoreload 2

import src.cnn.cnn_gradcam as gradcam
from src.cnn.cnn_registry import build_model
from src.cnn.cnn_utils import get_device, load_datasets
from src.cnn.configs import ExperimentConfig

In [ ]:
# optional: override the auto "last conv layer" choice
target_layers = None
# examples if you want explicit control:
target_layers = [model.features[3]]  # for shallow_model -> conv2
print(target_layers)

In [ ]:
# 1) chosen image + chosen class
heatmap_img = gradcam.show_heatmap_on_image(
    model=model,
    dataset=val_dataset,
    index=347,
    target_class=class_names.index("cat"),
    class_names=class_names,
    mean = [0.0, 0.0, 0.0], # standard values, if no normalization was done
    std = [1.0, 1.0, 1.0],
    target_layers=target_layers,
    method="gradcam",
    device=cfg.train.device,
)

heatmap_img = gradcam.show_heatmap_on_image(
    model=model,
    dataset=val_dataset,
    index=3409,
    target_class=class_names.index("cat"),
    class_names=class_names,
    mean = [0.0, 0.0, 0.0], # standard values, if no normalization was done
    std = [1.0, 1.0, 1.0],
    target_layers=target_layers,
    method="gradcam",
    device=cfg.train.device,
)

heatmap_img = gradcam.show_heatmap_on_image(
    model=model,
    dataset=val_dataset,
    index=3409,
    target_class=class_names.index("horse"),
    class_names=class_names,
    mean = [0.0, 0.0, 0.0], # standard values, if no normalization was done
    std = [1.0, 1.0, 1.0],
    target_layers=target_layers,
    method="gradcam",
    device=cfg.train.device,
)

In [ ]:
# 2) chosen image + all class heatmaps
heatmaps_img = gradcam.show_heatmaps_for_all_classes_on_image(
    model=model,
    dataset=val_dataset,
    index=678,
    class_names=class_names,
    mean = [0.0, 0.0, 0.0], # standard values, if no normalization was done
    std = [1.0, 1.0, 1.0],
    target_layers=target_layers,
    method="gradcam",
    device=cfg.train.device,
)

In [ ]:
# 3) fixed target class + one random image per true class from VAL
heatmaps_imgs = gradcam.show_target_class_across_true_classes(
    model=model,
    dataset=val_dataset,
    target_class=class_names.index("zebra"),
    class_names=class_names,
    mean = [0.0, 0.0, 0.0], # standard values, if no normalization was done
    std = [1.0, 1.0, 1.0],
    seed=62,
    target_layers=target_layers,
    method="gradcam",
    device=cfg.train.device,
)